# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant-python) library. The data schema is described using Croissant and accessed via its schema URL.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Ensure mlcroissant is installed
!pip install --quiet mlcroissant

## 1. Data Loading

Load dataset metadata and records using mlcroissant.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'
# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Print basic information
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview

Review available record sets (`@id`), each field's `@id`, and associated columns. This helps to know how to reference elements programmatically.

In [ ]:
# Enumerate all record sets and their fields with @ids
from collections import defaultdict

record_set_ids = []
overview = defaultdict(dict)

print("Available Record Sets and Fields:")
for record_set in dataset.metadata.record_sets:
    rs_id = record_set.id
    record_set_ids.append(rs_id)
    print(f"- RecordSet: {rs_id} | Name: {getattr(record_set, 'name', '')}")
    fields = getattr(record_set, 'fields', []) if hasattr(record_set, 'fields') else []
    col_ids = []
    for field in fields:
        print(f"    - Field: {field.id} | Name: {getattr(field, 'name', '')} | DataType: {getattr(field, 'data_type', '')}")
        col = getattr(field, 'column', None)
        if col:
            print(f"        - Column: {col.id} | Name: {getattr(col, 'name', '')}")
            col_ids.append(col.id)
    overview[rs_id]['fields'] = [f.id for f in fields]
    overview[rs_id]['columns'] = col_ids
    print()
if not record_set_ids:
    print('No record sets found in this dataset! (double-check the Croissant schema)')

## 3. Data Extraction

Load data for the main record set into Pandas DataFrame. All recordset and field references are by their full `@id`.

In [ ]:
# We'll assume only one main record set for this tabular clinical dataset
if not record_set_ids:
    raise ValueError('No record sets were found in the dataset metadata!')

# Use the first (and likely only) record set
main_record_set_id = record_set_ids[0]

print(f"Extracting records for RecordSet: {main_record_set_id}")
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
print(f"Loaded DataFrame with {df.shape[0]} rows and columns:\n{df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)

Common EDA steps, such as filtering and normalizing numeric fields, and grouping by key attributes by referencing field `@id`s as column names.

We will identify a numeric field (e.g., age) by looking at field data types above.

In [ ]:
# Find a likely numeric field from overview (for example, 'Age')
numeric_field_id = None
group_field_id = None

# We will heuristically pick columns with 'age' or that have integer/float datatype
for record_set in dataset.metadata.record_sets:
    if record_set.id == main_record_set_id:
        for field in getattr(record_set, 'fields', []):
            if 'age' in field.id.lower() or 'age' in getattr(field, 'name', '').lower():
                numeric_field_id = field.id
            if 'sex' in field.id.lower() or 'gender' in field.id.lower():
                group_field_id = field.id
        # If we did not find from name, try by type
        if not numeric_field_id:
            for field in getattr(record_set, 'fields', []):
                if getattr(field, 'data_type', None) in ('Number', 'Integer', 'Float'):
                    numeric_field_id = field.id
                    break

if numeric_field_id is None:
    raise ValueError('Could not automatically infer a numeric field for demo!')
if group_field_id is None:
    # Use the first string/categorical-like field (excluding @id and the numeric field)
    for col in df.columns:
        if col not in ('@id', numeric_field_id) and df[col].dtype == object:
            group_field_id = col
            break
    else:
        group_field_id = numeric_field_id  # fallback, will not actually group

print(f"Numeric field used: {numeric_field_id}")
print(f"Grouping field: {group_field_id}")

# Convert to numeric (if necessary)
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Apply filter: show records where numeric_field > 10
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold} (Total: {len(filtered_df)}):\n")
display(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().rename(columns={numeric_field_id: f"mean_{numeric_field_id}"})
    print(f"\nMean {numeric_field_id} by {group_field_id}:")
    display(grouped_df.head())

## 5. Visualization

Visualize the distribution of the numeric variable and its breakdown by group if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=10)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot by group (if grouping meaningful and >1 group)
if group_field_id in df.columns and df[group_field_id].nunique() > 1:
    plt.figure(figsize=(10,4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

In this notebook, you learned to explore a FAIR^2 clinical dataset described using the Croissant metadata standard and accessed via the `mlcroissant` library. You:
- Listed the available record sets and fields by their `@id`s,
- Loaded data into pandas for analysis,
- Filtered and normalized a numeric variable using its Croissant field `@id`,
- Grouped and visualized data for exploratory insight.

**Next steps:** Investigate additional fields, relationships, or take dataset-specific analysis questions further!
